# Problem 99: The Electric Vehicle Routing Problem (EVRP)

## Tier 1: Mixed-Integer Programming Formulation

### Key Assumptions
- Electric vehicles have limited battery capacity that must be respected
- Charging stations are available at known locations with fixed charging rates
- Energy consumption depends on distance traveled between nodes
- Each customer must be visited exactly once by one vehicle
- All vehicles start and end at the depot

### Approach (Step-by-Step)

The Electric Vehicle Routing Problem (EVRP) extends the classic VRP by incorporating battery constraints and charging station visits. We'll use Mixed-Integer Programming (MIP) to formulate this problem mathematically:

1. **Define Sets and Parameters**: Create sets for nodes (customers, depot, charging stations) and vehicles
2. **Decision Variables**: Binary variables for routing, continuous variables for battery level and time
3. **Objective Function**: Minimize total travel distance
4. **Constraints**: Ensure routing feasibility, battery management, and time windows
5. **Solve**: Use pulp solver to find optimal solution for small instances

### What to Look for in the Results
- Optimal routes that respect battery constraints
- Charging station insertions when needed
- Total distance and energy consumption
- Feasibility verification (no battery violations)

### Concrete Example

We'll solve a small instance with:
- 1 depot
- 3 customers
- 2 charging stations
- 2 electric vehicles
- Battery capacity: 100 kWh
- Energy consumption: 0.8 kWh per km

In [ ]:
# Import required libraries for MIP formulation
import pulp
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from dataclasses import dataclass
from typing import List, Tuple, Dict
import warnings
warnings.filterwarnings('ignore')

# Set style for professional visualizations
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

In [ ]:
@dataclass
class Node:
    """Represents a node in the EVRP network"""
    id: int
    x: float
    y: float
    node_type: str  # 'depot', 'customer', 'charging_station'
    service_time: float = 0.0
    
@dataclass
class Vehicle:
    """Represents an electric vehicle"""
    id: int
    battery_capacity: float  # kWh
    initial_charge: float  # kWh
    
@dataclass
class EVRPInstance:
    """EVRP problem instance"""
    nodes: List[Node]
    vehicles: List[Vehicle]
    energy_consumption_rate: float  # kWh per km
    charging_rate: float  # kW
    
def create_example_instance():
    """Create the concrete example from the problem description"""
    
    # Define nodes: depot (0), customers (1,2,3), charging stations (4,5)
    nodes = [
        Node(0, 0, 0, 'depot'),  # Depot at origin
        Node(1, 10, 5, 'customer', service_time=0.5),   # Customer 1
        Node(2, 8, 12, 'customer', service_time=0.5),   # Customer 2
        Node(3, 15, 8, 'customer', service_time=0.5),   # Customer 3
        Node(4, 5, 8, 'charging_station'),  # Charging Station 1
        Node(5, 12, 2, 'charging_station'),  # Charging Station 2
    ]
    
    # Define vehicles
    vehicles = [
        Vehicle(0, battery_capacity=100, initial_charge=100),
        Vehicle(1, battery_capacity=100, initial_charge=100)
    ]
    
    return EVRPInstance(
        nodes=nodes,
        vehicles=vehicles,
        energy_consumption_rate=0.8,  # 0.8 kWh per km
        charging_rate=50.0  # 50 kW charging rate
    )

# Create the instance
instance = create_example_instance()
print(f"Created EVRP instance with {len(instance.nodes)} nodes and {len(instance.vehicles)} vehicles")

In [ ]:
def calculate_distance(node1: Node, node2: Node) -> float:
    """Calculate Euclidean distance between two nodes"""
    return np.sqrt((node1.x - node2.x)**2 + (node1.y - node2.y)**2)

def calculate_energy_consumption(distance: float, rate: float) -> float:
    """Calculate energy consumption for a given distance"""
    return distance * rate

# Pre-calculate distances and energy consumption
n_nodes = len(instance.nodes)
n_vehicles = len(instance.vehicles)

# Distance matrix
distances = np.zeros((n_nodes, n_nodes))
energy_consumption = np.zeros((n_nodes, n_nodes))

for i in range(n_nodes):
    for j in range(n_nodes):
        if i != j:
            distances[i, j] = calculate_distance(instance.nodes[i], instance.nodes[j])
            energy_consumption[i, j] = calculate_energy_consumption(distances[i, j], instance.energy_consumption_rate)

print("Distance Matrix (km):")
print(np.round(distances, 2))
print("\nEnergy Consumption Matrix (kWh):")
print(np.round(energy_consumption, 2))

In [ ]:
def solve_evrp_mip(instance: EVRPInstance):
    """Solve EVRP using Mixed-Integer Programming"""
    
    # Create the optimization problem
    model = pulp.LpProblem("EVRP_MIP", pulp.LpMinimize)
    
    # Define sets
    N = range(len(instance.nodes))  # All nodes
    C = [i for i, node in enumerate(instance.nodes) if node.node_type == 'customer']  # Customers
    F = [i for i, node in enumerate(instance.nodes) if node.node_type == 'charging_station']  # Charging stations
    V = range(len(instance.vehicles))  # Vehicles
    depot = 0  # Depot node index
    
    # Decision variables
    # x[i,j,k] = 1 if vehicle k travels from node i to node j
    x = pulp.LpVariable.dicts('x', [(i, j, k) for i in N for j in N for k in V if i != j], 
                            cat='Binary')
    
    # u[i,k] = battery level of vehicle k upon arrival at node i
    u = pulp.LpVariable.dicts('u', [(i, k) for i in N for k in V], 
                            lowBound=0, upBound=instance.vehicles[0].battery_capacity)
    
    # y[i,k] = amount of energy charged at node i for vehicle k
    y = pulp.LpVariable.dicts('y', [(i, k) for i in F for k in V], 
                            lowBound=0, upBound=instance.vehicles[0].battery_capacity)
    
    # Objective function: minimize total distance
    model += pulp.lpSum(distances[i, j] * x[i, j, k] for i in N for j in N for k in V if i != j)
    
    # Constraints
    # Each customer must be visited exactly once
    for i in C:
        model += pulp.lpSum(x[j, i, k] for j in N for k in V if j != i) == 1
    
    # Flow conservation
    for i in C + F:
        for k in V:
            model += (pulp.lpSum(x[j, i, k] for j in N if j != i) == 
                     pulp.lpSum(x[i, j, k] for j in N if j != i))
    
    # Each vehicle leaves depot at most once
    for k in V:
        model += pulp.lpSum(x[depot, j, k] for j in N if j != depot) <= 1
    
    # Routes start and end at depot
    for k in V:
        model += (pulp.lpSum(x[i, depot, k] for i in N if i != depot) == 
                 pulp.lpSum(x[depot, j, k] for j in N if j != depot))
    
    # Battery consumption constraints
    M = 1000  # Big-M constant
    for i in N:
        for j in N:
            for k in V:
                if i != j:
                    model += (u[j, k] <= u[i, k] - energy_consumption[i, j] + 
                             M * (1 - x[i, j, k]) + 
                             (y[i, k] if i in F else 0))
    
    # Initial battery level at depot
    for k in V:
        model += u[depot, k] == instance.vehicles[k].initial_charge
    
    # No charging at customers or depot
    for i in [depot] + C:
        for k in V:
            if i in F:
                model += y[i, k] == 0
    
    # Charging limits at charging stations
    for i in F:
        for k in V:
            model += y[i, k] <= instance.vehicles[0].battery_capacity - u[i, k]
    
    # Solve the model
    print("Solving MIP model...")
    model.solve(pulp.PULP_CBC_CMD(msg=0, timeLimit=60))
    
    return model, x, u, y

# Solve the EVRP
model, x_vars, u_vars, y_vars = solve_evrp_mip(instance)

print(f"\nOptimization Status: {pulp.LpStatus[model.status]}")
print(f"Objective Value (Total Distance): {pulp.value(model.objective):.2f} km")

In [ ]:
def extract_solution(x_vars, u_vars, y_vars, instance):
    """Extract and format the solution from MIP variables"""
    
    solution = {
        'routes': [],
        'total_distance': 0,
        'battery_levels': {},
        'charging_actions': {}
    }
    
    # Extract routes
    for k in range(len(instance.vehicles)):
        route = []
        current = 0  # Start at depot
        
        # Find next node from current position
        while True:
            found_next = False
            for j in range(len(instance.nodes)):
                if current != j and pulp.value(x_vars[current, j, k]) == 1:
                    route.append((current, j))
                    current = j
                    found_next = True
                    break
            
            if not found_next or current == 0:  # Back at depot or no more moves
                break
        
        if route:  # Only add non-empty routes
            solution['routes'].append(route)
    
    # Extract battery levels and charging
    for i in range(len(instance.nodes)):
        for k in range(len(instance.vehicles)):
            if pulp.value(u_vars[i, k]) is not None:
                solution['battery_levels'][(i, k)] = pulp.value(u_vars[i, k])
            
            if i in [idx for idx, node in enumerate(instance.nodes) if node.node_type == 'charging_station']:
                if pulp.value(y_vars[i, k]) is not None and pulp.value(y_vars[i, k]) > 0:
                    solution['charging_actions'][(i, k)] = pulp.value(y_vars[i, k])
    
    # Calculate total distance
    for route in solution['routes']:
        for i, j in route:
            solution['total_distance'] += distances[i, j]
    
    return solution

# Extract solution
solution = extract_solution(x_vars, u_vars, y_vars, instance)

print("Solution Summary:")
print(f"Total Distance: {solution['total_distance']:.2f} km")
print(f"Number of routes: {len(solution['routes'])}")
print(f"Charging actions: {len(solution['charging_actions'])}")

for k, route in enumerate(solution['routes']):
    print(f"\nRoute {k+1}:")
    route_str = "Depot"
    for i, j in route:
        node_type = instance.nodes[j].node_type
        if node_type == 'customer':
            route_str += f" -> Customer {j}"
        elif node_type == 'charging_station':
            route_str += f" -> Charging Station {j}"
    print(route_str)

In [ ]:
def visualize_solution(instance, solution):
    """Create comprehensive visualization of the EVRP solution"""
    
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle('Electric Vehicle Routing Problem - MIP Solution Analysis', fontsize=16, fontweight='bold')
    
    # Plot 1: Route visualization
    colors = ['blue', 'red', 'green', 'orange']
    
    # Plot nodes
    for i, node in enumerate(instance.nodes):
        if node.node_type == 'depot':
            ax1.scatter(node.x, node.y, s=300, c='black', marker='s', label='Depot', zorder=5)
            ax1.annotate('DEPOT', (node.x, node.y), xytext=(5, 5), textcoords='offset points', fontweight='bold')
        elif node.node_type == 'customer':
            ax1.scatter(node.x, node.y, s=200, c='blue', marker='o', label='Customer' if i == 1 else '', zorder=4)
            ax1.annotate(f'C{i}', (node.x, node.y), xytext=(5, 5), textcoords='offset points')
        else:  # charging station
            ax1.scatter(node.x, node.y, s=200, c='green', marker='^', label='Charging Station' if i == 4 else '', zorder=4)
            ax1.annotate(f'CS{i}', (node.x, node.y), xytext=(5, 5), textcoords='offset points')
    
    # Plot routes
    for k, route in enumerate(solution['routes']):
        for i, j in route:
            ax1.plot([instance.nodes[i].x, instance.nodes[j].x], 
                    [instance.nodes[i].y, instance.nodes[j].y], 
                    color=colors[k % len(colors)], linewidth=2, alpha=0.7)
    
    ax1.set_xlabel('X Coordinate (km)')
    ax1.set_ylabel('Y Coordinate (km)')
    ax1.set_title('Optimal EV Routes with Charging Stations')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Plot 2: Battery levels throughout routes
    for k, route in enumerate(solution['routes']):
        battery_levels = []
        positions = []
        
        for i, j in route:
            if (i, k) in solution['battery_levels']:
                battery_levels.append(solution['battery_levels'][(i, k)])
                positions.append(f"Node {i}")
        
        if battery_levels:
            ax2.plot(range(len(battery_levels)), battery_levels, 
                   marker='o', label=f'Vehicle {k+1}', linewidth=2)
    
    ax2.axhline(y=20, color='red', linestyle='--', alpha=0.5, label='Safety Threshold (20 kWh)')
    ax2.set_xlabel('Route Position')
    ax2.set_ylabel('Battery Level (kWh)')
    ax2.set_title('Battery Levels Throughout Routes')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # Plot 3: Distance breakdown by route
    route_distances = []
    route_labels = []
    
    for k, route in enumerate(solution['routes']):
        route_dist = 0
        for i, j in route:
            route_dist += distances[i, j]
        route_distances.append(route_dist)
        route_labels.append(f'Route {k+1}')
    
    bars = ax3.bar(route_labels, route_distances, color=colors[:len(route_distances)], alpha=0.7)
    ax3.set_ylabel('Distance (km)')
    ax3.set_title('Distance Breakdown by Route')
    ax3.grid(True, alpha=0.3, axis='y')
    
    # Add value labels on bars
    for bar, dist in zip(bars, route_distances):
        height = bar.get_height()
        ax3.text(bar.get_x() + bar.get_width()/2., height + 0.5,
                f'{dist:.1f} km', ha='center', va='bottom')
    
    # Plot 4: Solution statistics
    stats_text = f"""
    Solution Statistics:
    ─────────────────────
    Total Distance: {solution['total_distance']:.2f} km
    Total Energy: {solution['total_distance'] * instance.energy_consumption_rate:.2f} kWh
    Routes Used: {len(solution['routes'])}
    Charging Stops: {len(solution['charging_actions'])}
    Customers Served: {len([n for n in instance.nodes if n.node_type == 'customer'])}
    
    Vehicle Specifications:
    ─────────────────────
    Battery Capacity: {instance.vehicles[0].battery_capacity} kWh
    Energy Consumption: {instance.energy_consumption_rate} kWh/km
    Charging Rate: {instance.charging_rate} kW
    """
    
    ax4.text(0.1, 0.5, stats_text, transform=ax4.transAxes, fontsize=11,
             verticalalignment='center', fontfamily='monospace',
             bbox=dict(boxstyle='round', facecolor='lightgray', alpha=0.8))
    ax4.set_xlim(0, 1)
    ax4.set_ylim(0, 1)
    ax4.axis('off')
    ax4.set_title('Solution Summary', fontweight='bold')
    
    plt.tight_layout()
    plt.show()

# Visualize the solution
visualize_solution(instance, solution)

In [ ]:
def verify_feasibility(instance, solution):
    """Verify that the solution is feasible"""
    
    print("Feasibility Verification:")
    print("=" * 50)
    
    # Check 1: All customers visited exactly once
    customers_visited = {}
    for route in solution['routes']:
        for i, j in route:
            if instance.nodes[j].node_type == 'customer':
                customers_visited[j] = customers_visited.get(j, 0) + 1
    
    print(f"✓ Customers visited exactly once: {all(count == 1 for count in customers_visited.values())}")
    
    # Check 2: Battery constraints respected
    battery_violations = 0
    for (i, k), battery_level in solution['battery_levels'].items():
        if battery_level < 0 or battery_level > instance.vehicles[0].battery_capacity:
            battery_violations += 1
    
    print(f"✓ Battery constraints respected: {battery_violations == 0}")
    
    # Check 3: Charging only at charging stations
    charging_violations = 0
    for (i, k), charge_amount in solution['charging_actions'].items():
        if instance.nodes[i].node_type != 'charging_station':
            charging_violations += 1
    
    print(f"✓ Charging only at stations: {charging_violations == 0}")
    
    # Check 4: Routes start and end at depot
    depot_violations = 0
    for route in solution['routes']:
        if route[0][0] != 0:  # First move should start from depot
            depot_violations += 1
        if route[-1][1] != 0:  # Last move should end at depot
            depot_violations += 1
    
    print(f"✓ Routes start/end at depot: {depot_violations == 0}")
    
    # Overall feasibility
    is_feasible = (all(count == 1 for count in customers_visited.values()) and 
                   battery_violations == 0 and 
                   charging_violations == 0 and 
                   depot_violations == 0)
    
    print(f"\n🎯 Overall Solution Feasible: {is_feasible}")
    
    return is_feasible

# Verify solution feasibility
feasible = verify_feasibility(instance, solution)

In [ ]:
def sensitivity_analysis(instance):
    """Perform sensitivity analysis on key parameters"""
    
    print("Sensitivity Analysis")
    print("=" * 50)
    
    # Test different battery capacities
    battery_capacities = [60, 80, 100, 120, 140]
    results = []
    
    for capacity in battery_capacities:
        # Create modified instance
        modified_instance = EVRPInstance(
            nodes=instance.nodes,
            vehicles=[Vehicle(0, battery_capacity=capacity, initial_charge=capacity),
                      Vehicle(1, battery_capacity=capacity, initial_charge=capacity)],
            energy_consumption_rate=instance.energy_consumption_rate,
            charging_rate=instance.charging_rate
        )
        
        try:
            # Solve with time limit for sensitivity analysis
            model, x_vars, u_vars, y_vars = solve_evrp_mip(modified_instance)
            
            if pulp.LpStatus[model.status] == 'Optimal':
                sol = extract_solution(x_vars, u_vars, y_vars, modified_instance)
                results.append({
                    'battery_capacity': capacity,
                    'total_distance': sol['total_distance'],
                    'charging_stops': len(sol['charging_actions']),
                    'status': 'Optimal'
                })
            else:
                results.append({
                    'battery_capacity': capacity,
                    'total_distance': float('inf'),
                    'charging_stops': 0,
                    'status': pulp.LpStatus[model.status]
                })
        except Exception as e:
            results.append({
                'battery_capacity': capacity,
                'total_distance': float('inf'),
                'charging_stops': 0,
                'status': 'Error'
            })
    
    # Display results
    print(f"{'Battery Capacity':<15} {'Total Distance':<15} {'Charging Stops':<15} {'Status':<10}")
    print("-" * 60)
    for result in results:
        if result['total_distance'] != float('inf'):
            print(f"{result['battery_capacity']:<15} {result['total_distance']:<15.2f} {result['charging_stops']:<15} {result['status']:<10}")
        else:
            print(f"{result['battery_capacity']:<15} {'N/A':<15} {result['charging_stops']:<15} {result['status']:<10}")
    
    return results

# Perform sensitivity analysis
sensitivity_results = sensitivity_analysis(instance)

### Why This Tier Exists vs Earlier Tiers

This Mixed-Integer Programming formulation serves as the mathematical foundation for the Electric Vehicle Routing Problem. It provides:

**Advantages:**
- **Guaranteed Optimality**: For small instances, MIP provides provably optimal solutions
- **Mathematical Rigor**: Precise formulation of all constraints and objectives
- **Benchmark Quality**: Serves as a gold standard for comparing heuristic methods
- **Complete Constraint Handling**: Explicitly models battery constraints, charging decisions, and routing logic

**Disadvantages:**
- **Computational Complexity**: Exponential time complexity makes it impractical for large instances
- **Scalability Issues**: Becomes intractable beyond 15-20 customers
- **Long Solution Times**: May require hours for medium-sized problems

**When to Use This Tier:**
- Small-scale problems (≤ 10 customers) where optimality is critical
- Academic research and validation of heuristic methods
- Benchmarking for developing new algorithms
- Problems where solution quality is more important than computation time

### Pros/Cons Summary

| Aspect | Pros | Cons |
|--------|------|------|
| Solution Quality | Optimal solutions guaranteed | May be slow for larger instances |
| Mathematical Clarity | Clear, formal problem definition | Complex formulation requires expertise |
| Constraint Handling | Complete and precise modeling | Difficult to extend with new constraints |
| Computational Requirements | Moderate for small instances | Exponential growth with problem size |
| Implementation | Well-supported by commercial solvers | Requires optimization software licenses |

This tier establishes the theoretical foundation and provides the optimal benchmark against which all subsequent heuristic and metaheuristic approaches will be compared.